# Task 1: Multi-Class Benchmark Classification Pipeline

**Dataset:** Iris (built into scikit-learn)

This notebook trains Logistic Regression and Decision Tree models, evaluates them, and saves metrics and plots. Run cells from top to bottom.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


## 1. Load and inspect data


In [ ]:
iris = load_iris(as_frame=True)
X = iris.data.copy()
y = iris.target.copy()
df = X.copy()
df['species'] = pd.Categorical.from_codes(y, iris.target_names)
print('Dataset shape:', df.shape)
display(df.head())
print('\nMissing values:\n', df.isnull().sum())
print('\nSpecies counts:\n', df['species'].value_counts())


## 2. Exploratory data analysis


In [ ]:
pair_grid = sns.pairplot(df, hue='species', diag_kind='hist')
pair_grid.fig.suptitle('Iris Feature Relationships by Species', y=1.02)
plt.show()


## 3. Train/test split (80/20)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'Training samples: {len(X_train)}')
print(f'Testing samples: {len(X_test)}')


## 4. Train models
Scaling is applied only to Logistic Regression and is fit on training data within the pipeline. Decision Trees do not need feature scaling.


In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE)
}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'Trained: {name}')


## 5. Evaluate models


In [ ]:
rows = []
predictions = {}
for name, model in models.items():
    pred = model.predict(X_test)
    predictions[name] = pred
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision (macro)': precision_score(y_test, pred, average='macro', zero_division=0),
        'Recall (macro)': recall_score(y_test, pred, average='macro', zero_division=0),
        'F1-score (macro)': f1_score(y_test, pred, average='macro', zero_division=0)
    })
    print(f'\\n{name} — classification report')
    print(classification_report(y_test, pred, target_names=iris.target_names, zero_division=0))
results_df = pd.DataFrame(rows).set_index('Model')
display(results_df.round(4))
results_df.to_csv(OUTPUT_DIR / 'model_comparison.csv')


## 6. Confusion matrices


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions[name], labels=np.arange(len(iris.target_names)),
        display_labels=iris.target_names, cmap='Blues', ax=ax, colorbar=False
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()
fig.savefig(OUTPUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')


## 7. Model comparison chart


In [ ]:
ax = results_df.plot(kind='bar', figsize=(10, 5), ylim=(0, 1.05))
ax.set_title('Model Performance Comparison')
ax.set_ylabel('Score')
ax.set_xlabel('Model')
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()
ax.get_figure().savefig(OUTPUT_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')


## 8. Optional decision boundary visualization
This plot uses only petal length and petal width to visualize class regions. The benchmark models above use all four features.


In [ ]:
feature_names = ['petal length (cm)', 'petal width (cm)']
X2 = X[feature_names]
X2_train, _, y2_train, _ = train_test_split(
    X2, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
boundary_models = {
    'Logistic Regression (2 features)': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Decision Tree (2 features)': DecisionTreeClassifier(random_state=RANDOM_STATE)
}
x_min, x_max = X2.iloc[:, 0].min() - 0.5, X2.iloc[:, 0].max() + 0.5
y_min, y_max = X2.iloc[:, 1].min() - 0.5, X2.iloc[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))
grid = pd.DataFrame({feature_names[0]: xx.ravel(), feature_names[1]: yy.ravel()})
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, model) in zip(axes, boundary_models.items()):
    model.fit(X2_train, y2_train)
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, levels=np.arange(-0.5, len(iris.target_names) + 0.5, 1))
    sns.scatterplot(x=X2[feature_names[0]], y=X2[feature_names[1]], hue=pd.Categorical.from_codes(y, iris.target_names), ax=ax, s=35, edgecolor='black')
    ax.set_title(name)
    ax.set_xlabel(feature_names[0])
    ax.set_ylabel(feature_names[1])
plt.tight_layout()
plt.show()
fig.savefig(OUTPUT_DIR / 'decision_boundaries.png', dpi=150, bbox_inches='tight')


## 9. Conclusion
Use the generated metric table to identify which model performs best on this particular test split. Avoid claiming one model is universally better based on a single split.

## Submission checklist
- [ ] Run every cell from top to bottom.
- [ ] Confirm the plots and metrics appear.
- [ ] Save the notebook with outputs.
- [ ] Include `outputs/model_comparison.csv` and generated charts in your repository.
- [ ] Add the repository README and submit the GitHub URL.
